In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\HAI\GUVI\Planetary_defense\regression\cleaned.csv")
df.head()

,neo,pha,H,diameter,albedo,epoch,epoch_mjd,epoch_cal,e,a,...,ma,ad,n,tp,tp_cal,per,per_y,moid_ld,class,rms
0,N,N,12.0,10.2425,0.0900,2459000.5,59000.0,20200531.0,0.076009,2.769165,...,77.372098,2.979647,0.213885,2.458239e+06,2.018043e+07,1683.145703,4.608202,620.640533,MBA,0.43301
1,N,N,12.0,10.2425,0.1010,2459000.5,59000.0,20200531.0,0.229972,2.773841,...,144.975675,3.411748,0.213345,2.458321e+06,2.018072e+07,1687.410992,4.619880,480.348639,MBA,0.40341
2,N,N,12.0,10.2425,0.2140,2459000.5,59000.0,20200531.0,0.256936,2.668285,...,125.435355,3.353865,0.226129,2.458446e+06,2.018112e+07,1592.013769,4.358696,402.514639,MBA,0.40341
3,N,N,12.0,10.2425,0.3905,2459000.5,59000.0,20200531.0,0.088721,2.361418,...,95.861938,2.570926,0.271609,2.458248e+06,2.018051e+07,1325.432763,3.628837,443.451432,MBA,0.40341
4,N,N,12.0,10.2425,0.2740,2459000.5,59000.0,20200531.0,0.190913,2.574037,...,17.846343,3.065455,0.238661,2.458926e+06,2.020032e+07,1508.414421,4.129814,426.433027,MBA,0.52191


In [2]:
obj_cols = df.select_dtypes(exclude="number").columns

for col in obj_cols:
    print(f"{col}:",df[col].value_counts())

neo: neo
N    135360
Y       849
Name: count, dtype: int64
pha: pha
N    135988
Y       221
Name: count, dtype: int64
class: class
MBA    124971
OMB      7521
TJN      1873
IMB       566
APO       486
MCA       358
AMO       268
ATE        95
CEN        51
TNO        12
AST         8
Name: count, dtype: int64


#### Encoding

In [4]:
from sklearn.preprocessing import OneHotEncoder

# Columns to encode
cat_cols = ['neo', 'pha', 'class']

# Initialize encoder
ohe = OneHotEncoder(
    sparse_output=False,
    handle_unknown='ignore'
)

# Fit and transform categorical columns
encoded_array = ohe.fit_transform(df[cat_cols])

# Convert encoded data into a DataFrame
encoded_df = pd.DataFrame(
    encoded_array,
    columns=ohe.get_feature_names_out(cat_cols),
    index=df.index
)

# Drop original categorical columns and add encoded columns
df = pd.concat(
    [df.drop(columns=cat_cols), encoded_df],
    axis=1
)
display(df.head())
print(df.shape)

,H,diameter,albedo,epoch,epoch_mjd,epoch_cal,e,a,i,om,...,class_APO,class_AST,class_ATE,class_CEN,class_IMB,class_MBA,class_MCA,class_OMB,class_TJN,class_TNO
0,12.0,10.2425,0.0900,2459000.5,59000.0,20200531.0,0.076009,2.769165,10.594067,80.305531,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,12.0,10.2425,0.1010,2459000.5,59000.0,20200531.0,0.229972,2.773841,26.543616,173.024741,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,12.0,10.2425,0.2140,2459000.5,59000.0,20200531.0,0.256936,2.668285,12.991043,169.851482,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,12.0,10.2425,0.3905,2459000.5,59000.0,20200531.0,0.088721,2.361418,7.141771,103.810804,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,12.0,10.2425,0.2740,2459000.5,59000.0,20200531.0,0.190913,2.574037,5.367427,141.571026,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


(136209, 35)


# Separate features and target

In [5]:
X = df.drop(columns=['diameter'])
y = df['diameter']

print("Features:", X.shape)
print("Target:", y.shape)

Features: (136209, 34)
Target: (136209,)


#### Train a Decision Tree and rank features

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

# Split the data first to avoid selecting features using test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Train Decision Tree Regressor
dt = DecisionTreeRegressor(
    max_depth=10,
    random_state=42
)

dt.fit(X_train, y_train)

# Get feature importance
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': dt.feature_importances_
})

# Sort by importance
importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
).reset_index(drop=True)

display(importance_df)

,Feature,Importance
0,H,0.664505
1,albedo,0.227806
2,n,0.090667
3,tp,0.008460
4,moid_ld,0.002415
5,tp_cal,0.001657
6,i,0.000904
7,ma,0.000855
8,per_y,0.000683
9,per,0.000427


#### Select the top 20 features

In [7]:
top_20_features = importance_df.head(20)['Feature'].tolist()

print("Top 20 features:")
print(top_20_features)

Top 20 features:
['H', 'albedo', 'n', 'tp', 'moid_ld', 'tp_cal', 'i', 'ma', 'per_y', 'per', 'ad', 'a', 'e', 'rms', 'w', 'om', 'neo_N', 'class_MBA', 'neo_Y', 'class_IMB']


#### Create the selected datasets

In [8]:
X_train_selected = X_train[top_20_features]
X_test_selected = X_test[top_20_features]

print("Training shape:", X_train_selected.shape)
print("Testing shape:", X_test_selected.shape)

Training shape: (108967, 20)
Testing shape: (27242, 20)


### Import the models and evaluation metrics`

In [9]:
import numpy as np

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

### Initialize the three models

In [10]:
models = {
    'Decision Tree': DecisionTreeRegressor(
        max_depth=10,
        random_state=42
    ),

    'Random Forest': RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ),

    'XGBoost': XGBRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1
    )
}

### Train and evaluate the models

In [11]:
results = []

for name, model in models.items():

    # Train the model
    model.fit(X_train_selected, y_train)

    # Make predictions
    y_pred = model.predict(X_test_selected)

    # Evaluate
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2 Score': r2
    })

results_df = pd.DataFrame(results)
print(results_df.sort_values(by='RMSE'))

           Model       MAE      RMSE  R2 Score
2        XGBoost  0.285244  0.430727  0.968855
1  Random Forest  0.282019  0.433207  0.968496
0  Decision Tree  0.323678  0.485217  0.960477
